# Dự án 3 — Đoán một con số, đoán một xác suất — Đề B: Điểm Toán từ các môn khác
## BẢN CỦA NHÓM

**Trường THCS và THPT Đinh Thiện Lý — Machine Learning Level 1 — Khối 10**

---

Dự án 1 nhóm đã thấy các môn “đi cùng nhau”. Giờ thử xa hơn: **biết điểm Văn và Ngoại ngữ, đoán điểm Toán được tới đâu?** 30 000 thí sinh thi tốt nghiệp THPT 2023, đã bỏ số báo danh.

**Câu hỏi lớn của dự án:** *Model đường thẳng và model xác suất giúp ra quyết định thế nào — và sai ở đâu?*

**Dữ liệu:** Kaggle — [Điểm thi THPT 2023](https://www.kaggle.com/datasets/duongtruongbinh/vietnamese-national-high-school-graduation-exam), CC0 (như Dự án 1)  

> Model đoán cho **nhóm thí sinh**, không dùng để đoán hay đánh giá một bạn cụ thể.

### Các mốc (nộp trên Canvas trước 23:59 Thứ 7 của tuần đó)

| Tuần | Mốc | Nộp |
|---|---|---|
| 1 · buổi 13 | **M1** Câu hỏi, dữ liệu đã kiểm tra, mốc đơn giản | Notebook bước 1 – 2 |
| 2 · buổi 14 | **M2** Model đầu tiên so với mốc | Notebook bước 3 |
| 3 · buổi 15 | **M3** Bảng thí nghiệm, chấm test một lần, 10 ca sai | Notebook bước 4 – 5 |
| 4 · buổi 16 | **M4** Poster A4 hoặc 2 slide, triển lãm, vấn đáp | Link poster + Nhật ký AI |

### Nhóm và vai — **đổi vai mỗi tuần**

Nhóm: … · Lớp: …

| Vai | Tuần 1 | Tuần 2 | Tuần 3 | Tuần 4 | Việc chính |
|---|---|---|---|---|---|
| Lái code | … | … | … | … | Gõ và chạy notebook; không ai gõ hộ hai tuần liền |
| Nghiên cứu | … | … | … | … | Đọc bài thư viện, hỏi Gemini “vì sao”, ghi Nhật ký AI |
| Thư ký số liệu | … | … | … | … | Ghi bảng thí nghiệm, kiểm tra con số trước khi báo cáo |
| Trình bày | … | … | … | … | Báo cáo 1 phút mỗi tuần, làm slide / poster |

### Quy tắc dùng AI

- **Được:** hỏi Gemini (✨ trong Colab) một lệnh làm gì, bấm *Giải thích lỗi*, hỏi *vì sao* làm bước này, nhờ gợi ý cách tiếp cận.
- **Không được:** nộp con số, biểu đồ, kết luận mà nhóm không tự chạy ra; để AI viết cả notebook.
- Ô code do AI gợi ý ghi dòng đầu `# AI gợi ý — nhóm đã kiểm tra: ...`. Mỗi tuần ghi **Nhật ký AI** ở cuối.
- Khi đi qua nhóm và ở mốc cuối, thầy gọi **một bạn bất kỳ** giải thích một dòng code hoặc một con số.

### Dữ liệu

| Cột | Ý nghĩa |
|---|---|
| `tinh` | Tỉnh, thành (theo mã sở GD&ĐT) |
| `toan` | Điểm Toán — **nhãn** |
| `van, ngoai_ngu` | Văn, Ngoại ngữ |
| `ly, hoa, sinh` | Tổ hợp KHTN (trống nếu không thi) |
| `su, dia, gdcd` | Tổ hợp KHXH (trống nếu không thi) |


---
# TUẦN 1 (buổi 13) — Câu hỏi, dữ liệu, mốc

**Mốc của tuần:** **M1** Câu hỏi, dữ liệu đã kiểm tra, mốc đơn giản  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 15 Hồi quy tuyến tính](https://vuongndlst.github.io/ml-level1/bai15/)


---
## Bước 0 — Nạp dữ liệu (đã viết sẵn, chỉ cần chạy)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
URL = "https://vuongndlst.github.io/ml-level1/du-lieu/"
TEP = "diem_thi_2023.csv"
# Đã tải file lên Colab thì đọc file; chưa thì đọc thẳng từ web khoá học
df = pd.read_csv(TEP if os.path.exists(TEP) else URL + TEP)
print(df.shape)
df.head()


### Câu hỏi của nhóm (M1)

*Gợi ý câu hỏi:* Môn nào đoán Toán tốt nhất? · Thêm tỉnh có giúp không? · Nhóm thi KHTN có dễ đoán hơn không?

| | Nhóm điền |
|---|---|
| Câu hỏi nhóm chọn | … |
| Cột sẽ dùng | … |
| Dự đoán trước (giả thuyết) | … |
| Làm sao biết dự đoán đúng hay sai? | … |


---
## Bước 1 — Khám phá và kiểm tra

> **🎯 Mục tiêu:** Biết nhãn là gì, tìm giá trị vô lý hoặc trùng trước khi học.
>
> **🧭 Vì sao bước này?** Chưa hiểu mỗi dòng, mỗi cột là gì thì không biết hỏi dữ liệu câu gì, cũng không biết cột nào là đáp án (nhãn) và cột nào dùng để đoán (feature).
>
> **📋 Nhiệm vụ:**
> 1. Xem thống kê các cột số: min, max có hợp lý không?
> 2. Đếm các dòng vô lý hoặc trùng.
> 3. Quyết định giữ hay bỏ, **ghi lý do**.
>
> **💡 Gợi ý lệnh:**
> - `df.describe()` — min, max, trung bình
> - `df.duplicated().sum()` — đếm dòng trùng
>
> **✅ Đạt khi:** nói được nhãn, số dòng vô lý / trùng và quyết định của nhóm.


In [ ]:
# Buoc 1 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 2 — Chia dữ liệu và mốc

> **🎯 Mục tiêu:** Chia train / validation / test 60 / 20 / 20; tính mốc (model lười) trên validation.
>
> **🧭 Vì sao bước này?** Model chỉ học trên tập huấn luyện; tập validation để so các thí nghiệm; tập test giữ kín tới cuối. Mốc (model lười) cũng chỉ học trên train — không có mốc thì không biết model giỏi hay bình thường.
>
> **📋 Nhiệm vụ:**
> 1. Bỏ các dòng nhóm đã quyết định bỏ ở bước 1.
> 2. Chia 3 tập, `random_state=42`.
> 3. Mốc: luôn đoán **giá trị trung bình** của train, tính MAE.
>
> **💡 Gợi ý lệnh:**
> - `train_test_split(df, test_size=0.2, random_state=42)` — tách test 20%
> - `mean_absolute_error(y_that, y_doan)` — MAE: sai trung bình, cùng đơn vị với nhãn
>
> **✅ Đạt khi:** có số dòng ba tập và con số mốc trên validation.


In [ ]:
# Buoc 2 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M1 (tuần 1)

- [ ] Bảng câu hỏi của nhóm đã điền
- [ ] Bước 1 – 2 có nhận xét kèm số, ghi rõ dòng nào bị bỏ và vì sao
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 2 (buổi 14) — Model đầu tiên

**Mốc của tuần:** **M2** Model đầu tiên so với mốc  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 15 Hồi quy tuyến tính](https://vuongndlst.github.io/ml-level1/bai15/)


---
## Bước 3 — Model đầu tiên — một cột

> **🎯 Mục tiêu:** Huấn luyện hồi quy tuyến tính với một cột, so với mốc, đọc model.
>
> **🧭 Vì sao bước này?** Model đầu tiên càng đơn giản càng tốt: một cột, một đường. Hiểu được nó rồi mới thêm cột.
>
> **📋 Nhiệm vụ:**
> 1. Huấn luyện trên train với **một** cột.
> 2. Tính MAE trên validation, so với mốc.
> 3. Đọc model: mỗi đơn vị thêm bao nhiêu (hệ số)?
>
> **💡 Gợi ý lệnh:**
> - `LinearRegression().fit(train[["..."]], train["..."])` — huấn luyện
> - `m1.coef_, m1.intercept_` — hệ số và hệ số tự do
>
> **✅ Đạt khi:** có con số so với mốc, một biểu đồ và một câu đọc model bằng lời.


In [ ]:
# Buoc 3 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M2 (tuần 2)

- [ ] Bước 3 có biểu đồ và nhận xét so với mốc
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 3 (buổi 15) — Thí nghiệm, test, ca sai

**Mốc của tuần:** **M3** Bảng thí nghiệm, chấm test một lần, 10 ca sai  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 15 Hồi quy tuyến tính](https://vuongndlst.github.io/ml-level1/bai15/), [Bài 16 Hồi quy logistic (ngưỡng)](https://vuongndlst.github.io/ml-level1/bai16/)


---
## Bước 4 — Bảng thí nghiệm

> **🎯 Mục tiêu:** Mỗi thí nghiệm đổi một thứ: thêm cột. Ghi kết quả validation vào bảng.
>
> **🧭 Vì sao bước này?** Mỗi thí nghiệm đổi **một** thứ (thêm một cột, đổi ngưỡng) và ghi vào bảng. So trên cùng tập validation.
>
> **📋 Nhiệm vụ:**
> 1. Thử ít nhất 3 thí nghiệm, mỗi lần đổi một thứ.
> 2. Cột chữ (quận, tỉnh) phải đổi thành cột 0 / 1 (`get_dummies`).
> 3. Chọn thí nghiệm tốt nhất **trên validation**.
>
> **💡 Gợi ý lệnh:**
> - `pd.get_dummies(d[cot], dtype=int)` — chữ → cột 0 / 1
>
> **✅ Đạt khi:** có bảng ≥ 3 thí nghiệm và thí nghiệm tốt nhất.


In [ ]:
# Buoc 4 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 5 — Chấm test và 10 ca sai

> **🎯 Mục tiêu:** Chấm test đúng một lần với thí nghiệm tốt nhất, đọc 10 ca sai nhiều nhất.
>
> **🧭 Vì sao bước này?** Chấm test đúng một lần, sau khi đã chốt thí nghiệm tốt nhất. Đọc 10 ca sai nhiều nhất để biết model yếu ở đâu.
>
> **📋 Nhiệm vụ:**
> 1. Huấn luyện lại với thí nghiệm tốt nhất, chấm trên test.
> 2. Xếp các ca theo độ sai, xem 10 ca đầu.
> 3. Các ca sai có điểm gì chung?
>
> **💡 Gợi ý lệnh:**
> - `sai.reindex(sai["lech"].abs().sort_values(ascending=False).index)` — xếp theo độ sai
>
> **✅ Đạt khi:** có con số test, 10 ca sai và một nhận xét về điểm chung.


In [ ]:
# Buoc 5 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M3 (tuần 3)

- [ ] Bảng thí nghiệm ≥ 3 dòng
- [ ] Con số test + 10 ca sai + nhận xét
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 4 (buổi 16) — Poster, triển lãm, vấn đáp

**Mốc của tuần:** **M4** Poster A4 hoặc 2 slide, triển lãm, vấn đáp  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** —


### Slide kết quả (M4) — một slide, 4 ô

| Ô | Nội dung |
|---|---|
| Câu hỏi | … |
| Một biểu đồ | … |
| Một con số | … |
| Thí nghiệm đắt nhất | … |
| Giới hạn | … |

*Làm trên Canva hoặc Google Slides; dán link vào ô nộp Canvas cùng link notebook.*


---
## Bước 6 — Nhật ký AI

Không dùng AI thì ghi *Không dùng*. Kiểm chứng = nhóm tự chạy lại, đối chiếu số, sửa chỗ AI sai.

| Nhóm hỏi AI | AI trả lời / viết | Nhóm kiểm chứng và sửa |
|---|---|---|
| … | … | … |
| … | … | … |
| … | … | … |


---
## Mở rộng — nhóm xong sớm (không bắt buộc)

1. Đoán **Văn** từ các môn khác: dễ hay khó hơn đoán Toán?
2. Hồi quy logistic: đoán “Toán từ 8 trở lên?” rồi đổi ngưỡng, như đề C.


In [ ]:
# Mo rong - code cua nhom



---
## Trước khi nộp

- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Mỗi bước có một ô nhận xét **có số và cỡ nhóm**
- [ ] Không có câu “… **gây ra** …” khi dữ liệu chỉ cho thấy đi cùng nhau
- [ ] Bảng vai trò và nhật ký AI đã điền
- [ ] Link notebook ở chế độ *Người xem*, dán lên Canvas cùng link Canva
